# LeetCode 3743 — Maximize Cyclic Partition Score bằng Python

- Link: https://leetcode.com/problems/maximize-cyclic-partition-score/
- Độ khó: **Hard**
- Chủ đề: DP phân hoạch, **nới lỏng (relaxation)** `max − min = max (a_i − a_j)`, DP máy trạng thái kiểu "mua/bán cổ phiếu k lần" (cả bán khống), mảng vòng — **cắt vòng tại phần tử lớn nhất**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumScore(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Viết DP phân hoạch trực tiếp `dp[t][i] = max_j dp[t−1][j] + range(j..i−1)` và thấy giới hạn của nó.
2. **Nới lỏng:** biên độ một đoạn `max − min` = giá trị lớn nhất của `a_x − a_y` với `x, y` trong đoạn ⇒ bài toán thành "chọn tối đa `k` **cặp** không đan xen, mỗi cặp góp `|a_x − a_y|`".
3. Giải bài cặp bằng **DP máy trạng thái** (tự do / đang giữ "mua" / đang giữ "bán khống") — giống *Best Time to Buy and Sell Stock IV* có bán khống — `O(n·k)`.
4. Xử lý **mảng vòng**: chứng minh chỉ cần thử hai cách cắt vòng ngay trước và ngay sau phần tử lớn nhất.
5. Kiểm chứng bằng vét cạn mọi cách cắt vòng.

## 2. Đề bài & ví dụ

`nums` là mảng **vòng**. Chia `nums` thành **tối đa** `k` mảng con (có thể vắt từ cuối về đầu). Biên độ của mảng con = `max − min`. Điểm = tổng biên độ. Tìm điểm lớn nhất.

**Ví dụ 1:** `nums = [1,2,3,3], k = 2` → `3` (`[2,3]` và `[3,1]` vắt vòng).

**Ví dụ 2:** `nums = [1,2,3,3], k = 1` → `2`.

**Ví dụ 3:** `nums = [1,2,3,3], k = 4` → `3`.

**Ràng buộc:** `1 ≤ n ≤ 1000`, `1 ≤ nums[i] ≤ 10^9`, `1 ≤ k ≤ n`.

In [1]:
from typing import List
from itertools import combinations
import random, time

examples = [([1, 2, 3, 3], 2, 3), ([1, 2, 3, 3], 1, 2), ([1, 2, 3, 3], 4, 3)]
for a, k, exp in examples:
    print(a, k, "-> kỳ vọng", exp)

[1, 2, 3, 3] 2 -> kỳ vọng 3
[1, 2, 3, 3] 1 -> kỳ vọng 2
[1, 2, 3, 3] 4 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Phân hoạch mảng vòng

Một phân hoạch vòng thành `r` đoạn tương ứng với `r` **vị trí cắt** trên vòng (cạnh giữa `i−1` và `i`). Với `r = 1`, cả vòng là một đoạn (điểm cắt không quan trọng — biên độ là `max − min` toàn mảng).

### 3.2. Biên độ là "cặp tốt nhất"

`max(S) − min(S) = max_{x, y ∈ S} (x − y)`. Vì vậy, với mỗi đoạn ta có thể "chọn" một phần tử mang dấu `+` và một phần tử mang dấu `−`; chọn tối ưu cho đúng biên độ, chọn bất kỳ cho giá trị **không vượt quá** biên độ.

In [2]:
S = [4, 9, 2, 7]
print("max - min =", max(S) - min(S), "| max cặp (x - y) =", max(x - y for x in S for y in S))

max - min = 7 | max cặp (x - y) = 7


### 3.3. Mua/bán cổ phiếu `k` lần (có bán khống)

Bài cổ phiếu: chọn tối đa `k` giao dịch không chồng lấn, mỗi giao dịch "mua ở `x`, bán ở `y` sau đó" lãi `a_y − a_x`. Nếu cho phép **bán khống** (bán trước, mua sau), mỗi giao dịch lãi `|a_y − a_x|`. Máy trạng thái: `free[j]` (đã xong `j` giao dịch, không giữ gì), `long[j]` (đang giữ lệnh mua), `short[j]` (đang giữ lệnh bán khống).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Bản tuyến tính.** Phân hoạch thành `≤ k` đoạn liên tiếp, tối đa tổng biên độ. Theo 3.2, điều này **bằng** bài: chọn `≤ k` cặp chỉ số, các cặp chiếm các **khoảng không giao nhau** (không đan xen), mỗi cặp góp `|a_x − a_y|`:

- (≤) Mỗi phân hoạch cho các cặp (max, min) trong từng đoạn — không đan xen.
- (≥) Mỗi tập cặp không đan xen ⇒ nới mỗi cặp thành một đoạn, các phần tử thừa gộp vào đoạn bên cạnh (biên độ chỉ tăng khi thêm phần tử) ⇒ điểm ≥ tổng cặp.

**Bản vòng.** Gọi `M` là vị trí phần tử **lớn nhất**. Trong phân hoạch tối ưu, đoạn chứa `M` có thể lấy `M` làm phần tử `+`. Cặp của nó `(M, m)` nằm **bên phải** hoặc **bên trái** `M` trong đoạn. Cắt vòng ngay **trước** `M` (mảng bắt đầu bằng `M`) hoặc ngay **sau** `M` (mảng kết thúc bằng `M`) thì cặp này và mọi cặp khác đều nằm trọn trong mảng tuyến tính ⇒ chỉ cần chạy bản tuyến tính hai lần.

## 5. Cách 1 — Brute Force: thử mọi tập vị trí cắt

In [3]:
def brute(nums, k):
    n = len(nums); best = 0
    for r in range(1, k + 1):
        for cuts in combinations(range(n), r):
            s = 0
            for t in range(r):
                st = cuts[t]
                en = cuts[t + 1] if t + 1 < r else cuts[0] + n
                seg = [nums[i % n] for i in range(st, en)]
                s += max(seg) - min(seg)
            best = max(best, s)
    return best

for a, k, exp in examples:
    print(brute(a, k), "kỳ vọng", exp)

3 kỳ vọng 3
2 kỳ vọng 2
3 kỳ vọng 3


**Nút thắt:** `O(2^n · n)` — chỉ cho `n ≤ 12`.

## 6. Ý tưởng thuật toán / bước đột phá

1. `M = argmax(nums)`.
2. `A1 = nums[M:] + nums[:M]` (bắt đầu bằng max), `A2 = nums[M+1:] + nums[:M+1]` (kết thúc bằng max).
3. Trên mỗi mảng, DP máy trạng thái với `j = 0..k` số cặp đã đóng:
   - `free'[j] = max(free[j], long[j−1] + x, short[j−1] − x)` (đóng một cặp tại `x`),
   - `long'[j] = max(long[j], free[j] − x)` (mở cặp với `x` là phần tử `−`),
   - `short'[j] = max(short[j], free[j] + x)` (mở cặp với `x` là phần tử `+`).
   Mỗi phần tử chỉ được dùng cho **một** hành động (vì mỗi phần tử thuộc đúng một đoạn).
4. Đáp án `max(max(free) của A1, max(free) của A2)`.

Độ phức tạp `O(n·k)`.

## 7. Các cách tiếp theo

- **Cách 2:** DP phân hoạch trực tiếp trên **mọi** phép quay — `O(n · n² · k)` (dùng để kiểm chứng ở cỡ vừa).
- **Cách 3:** nới lỏng + máy trạng thái trên **mọi** phép quay — `O(n² k)`.
- **Cách 4:** nới lỏng + máy trạng thái, **chỉ hai** phép quay quanh phần tử lớn nhất — `O(n k)` (lời giải nộp).

### 7.1. Cách 2 — DP phân hoạch trực tiếp, mọi phép quay

In [4]:
def partition_dp_linear(a, k):
    n = len(a); NEG = float("-inf")
    rng_ = [[0] * n for _ in range(n)]
    for i in range(n):
        mx = mn = a[i]
        for j in range(i, n):
            mx = max(mx, a[j]); mn = min(mn, a[j]); rng_[i][j] = mx - mn
    dp = [0] + [NEG] * n                       # dp[i] = tốt nhất cho tiền tố dài i với số đoạn hiện tại
    best = NEG
    for t in range(1, k + 1):
        nd = [NEG] * (n + 1)
        for i in range(1, n + 1):
            nd[i] = max((dp[j] + rng_[j][i - 1] for j in range(i) if dp[j] > NEG), default=NEG)
        dp = nd; best = max(best, dp[n])
    return best

def all_rotations_dp(nums, k):
    n = len(nums)
    return max(partition_dp_linear(nums[s:] + nums[:s], k) for s in range(n))

for a, k, exp in examples:
    print(all_rotations_dp(a, k), "kỳ vọng", exp)

3 kỳ vọng 3
2 kỳ vọng 2
3 kỳ vọng 3


**Vì sao đủ:** mọi phân hoạch vòng có ít nhất một vị trí cắt `s` ⇒ là một phân hoạch tuyến tính của phép quay bắt đầu tại `s`.

### 7.2. Cách 3 — Máy trạng thái trên mọi phép quay

In [5]:
NEG = float("-inf")

def pairs_linear(a, k):
    free = [0] + [NEG] * k
    lng = [NEG] * (k + 1)
    sht = [NEG] * (k + 1)
    for x in a:
        nf = [free[0]] + [max(free[j], lng[j - 1] + x, sht[j - 1] - x) for j in range(1, k + 1)]
        nl = [max(lng[j], free[j] - x) for j in range(k + 1)]
        ns = [max(sht[j], free[j] + x) for j in range(k + 1)]
        free, lng, sht = nf, nl, ns
    return max(free)

def all_rotations_pairs(nums, k):
    n = len(nums)
    return max(pairs_linear(nums[s:] + nums[:s], k) for s in range(n))

for a, k, exp in examples:
    print(all_rotations_pairs(a, k), "kỳ vọng", exp)

3 kỳ vọng 3
2 kỳ vọng 2
3 kỳ vọng 3


**Lưu ý:** cả ba danh sách mới đều tính từ trạng thái **cũ** (`free`, `lng`, `sht` trước khi xét `x`) ⇒ mỗi phần tử chỉ tham gia một hành động.

### 7.3. Cách 4 — Chỉ hai phép quay quanh max (lời giải chính)

In [6]:
def two_rotations(nums, k):
    m = nums.index(max(nums))
    a1 = nums[m:] + nums[:m]
    a2 = nums[m + 1:] + nums[:m + 1]
    return max(pairs_linear(a1, k), pairs_linear(a2, k))

for a, k, exp in examples:
    print(two_rotations(a, k), "kỳ vọng", exp)

3 kỳ vọng 3
2 kỳ vọng 2
3 kỳ vọng 3


## 8. TRACE từng bước

In [7]:
def trace_pairs(a, k):
    free = [0] + [NEG] * k; lng = [NEG] * (k + 1); sht = [NEG] * (k + 1)
    fmt = lambda v: "  -" if v == NEG else f"{v:>3}"
    print("mảng:", a, "| k =", k)
    for x in a:
        nf = [free[0]] + [max(free[j], lng[j - 1] + x, sht[j - 1] - x) for j in range(1, k + 1)]
        nl = [max(lng[j], free[j] - x) for j in range(k + 1)]
        ns = [max(sht[j], free[j] + x) for j in range(k + 1)]
        free, lng, sht = nf, nl, ns
        print(f"x={x}: free={[fmt(v) for v in free]} long={[fmt(v) for v in lng]} short={[fmt(v) for v in sht]}")
    print("max(free) =", max(free))
    return max(free)

nums = [1, 2, 3, 3]; m = nums.index(max(nums))
print("cắt trước max:"); trace_pairs(nums[m:] + nums[:m], 2)
print("cắt sau max:");  trace_pairs(nums[m + 1:] + nums[:m + 1], 2)

cắt trước max:
mảng: [3, 3, 1, 2] | k = 2
x=3: free=['  0', '  -', '  -'] long=[' -3', '  -', '  -'] short=['  3', '  -', '  -']
x=3: free=['  0', '  0', '  -'] long=[' -3', '  -', '  -'] short=['  3', '  -', '  -']
x=1: free=['  0', '  2', '  -'] long=[' -1', ' -1', '  -'] short=['  3', '  1', '  -']
x=2: free=['  0', '  2', '  1'] long=[' -1', '  0', '  -'] short=['  3', '  4', '  -']
max(free) = 2
cắt sau max:
mảng: [3, 1, 2, 3] | k = 2
x=3: free=['  0', '  -', '  -'] long=[' -3', '  -', '  -'] short=['  3', '  -', '  -']
x=1: free=['  0', '  2', '  -'] long=[' -1', '  -', '  -'] short=['  3', '  -', '  -']
x=2: free=['  0', '  2', '  -'] long=[' -1', '  0', '  -'] short=['  3', '  4', '  -']
x=3: free=['  0', '  2', '  3'] long=[' -1', '  0', '  -'] short=['  3', '  5', '  -']
max(free) = 3


3

In [8]:
# Ví dụ cần vắt vòng: tìm (có seed) một mảng mà bản tuyến tính KHÔNG đạt tối ưu vòng
rng_w = random.Random(21)
while True:
    n = rng_w.randint(2, 6); kk = rng_w.randint(1, n); a = [rng_w.randint(1, 6) for _ in range(n)]
    if pairs_linear(a, kk) < brute(a, kk):
        break
print("mảng:", a, "k =", kk, "| brute (vòng):", brute(a, kk), "| tuyến tính không quay:", pairs_linear(a, kk),
      "| hai phép quay:", two_rotations(a, kk))
assert two_rotations(a, kk) == brute(a, kk) > pairs_linear(a, kk)

mảng: [2, 1, 1, 3, 5, 4] k = 5 | brute (vòng): 7 | tuyến tính không quay: 5 | hai phép quay: 7


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maximumScore(self, nums: List[int], k: int) -> int:
        NEG = float("-inf")

        def linear(a):
            free = [0] + [NEG] * k
            lng = [NEG] * (k + 1)
            sht = [NEG] * (k + 1)
            rng = range(1, k + 1)
            for x in a:
                nf = [0] + [max(free[j], lng[j - 1] + x, sht[j - 1] - x) for j in rng]
                lng = [max(l, f - x) for l, f in zip(lng, free)]
                sht = [max(s, f + x) for s, f in zip(sht, free)]
                free = nf
            return max(free)

        m = nums.index(max(nums))
        return max(linear(nums[m:] + nums[:m]), linear(nums[m + 1:] + nums[:m + 1]))

sol = Solution()
for a, k, exp in examples:
    got = sol.maximumScore(a, k)
    print(a, k, "->", got, "✔" if got == exp else "✘")

[1, 2, 3, 3] 2 -> 3 ✔
[1, 2, 3, 3] 1 -> 2 ✔
[1, 2, 3, 3] 4 -> 3 ✔


## 10. Vì sao thuật toán đúng

1. **Tuyến tính ⇔ cặp không đan xen** (mục 4): hai chiều bất đẳng thức đã chứng minh; thêm phần tử vào một đoạn không làm giảm biên độ nên các phần tử "thừa" không gây hại.
2. **Máy trạng thái** liệt kê đúng mọi tập `≤ k` cặp không đan xen: duyệt trái sang phải, tại mỗi thời điểm có tối đa một cặp "đang mở"; mở bằng `−x` (sẽ đóng bằng `+y`) hoặc `+x` (đóng bằng `−y`); `free[j]` chỉ nhận cặp đã đóng. Do tính từ trạng thái cũ, một phần tử không vừa đóng vừa mở.
3. **Cắt vòng tại max.** Lấy phân hoạch vòng tối ưu `P` và đoạn `S ∋ M`. Chọn cặp của `S` là `(M, min S)`, các đoạn khác chọn (max, min). Nếu `min S` nằm sau `M` trong `S` (theo chiều vòng), cắt vòng ngay trước `M`: phần của `S` từ `M` tới `min S` là một khoảng ở đầu mảng `A1`, mọi đoạn khác nằm trọn trong `A1` (phần đầu của `S` trước `M` rơi vào cuối `A1`, không cần dùng). Ngược lại cắt ngay sau `M` (mảng `A2`). Vậy `max(lin(A1), lin(A2)) ≥ score(P)`; chiều ngược lại hiển nhiên vì phân hoạch tuyến tính của một phép quay là phân hoạch vòng. Trường hợp `S` là cả vòng (`r = 1`) cũng được bao.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn cắt | `O(2^n · n)` | `O(n)` |
| 2. DP phân hoạch × mọi phép quay | `O(n³ k)` | `O(n²)` |
| 3. Máy trạng thái × mọi phép quay | `O(n² k)` | `O(k)` |
| 4. Máy trạng thái × 2 phép quay | `O(n k)` | `O(k)` |

## 12. Bộ test

In [10]:
METHODS = [brute, all_rotations_dp, all_rotations_pairs, two_rotations, lambda a, k: Solution().maximumScore(a, k)]
for a, k, exp in examples:
    assert all(f(a, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().maximumScore([7], 1) == 0
assert Solution().maximumScore([5, 5, 5], 3) == 0
assert Solution().maximumScore([1, 9], 2) == 8                 # chỉ một đoạn có ích
assert Solution().maximumScore([1, 9, 1, 9], 2) == 16 == brute([1, 9, 1, 9], 2)
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên: so với vét cạn và các cách chậm
rng = random.Random(3743)
for _ in range(1500):
    n = rng.randint(1, 8); k = rng.randint(1, n)
    a = [rng.randint(1, 6) for _ in range(n)]
    b = brute(a, k)
    assert all(f(a, k) == b for f in METHODS[1:]), (a, k)
print("1500 test ngẫu nhiên: OK")

1500 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn: n = k = 1000
rng = random.Random(7)
a = [rng.randint(1, 10**9) for _ in range(1000)]
for k in (1, 37, 1000):
    t0 = time.time(); x = Solution().maximumScore(a, k); t1 = time.time()
    print(f"k = {k}: điểm {x}, thời gian {t1 - t0:.2f}s")
assert Solution().maximumScore(a, 1) == max(a) - min(a)

k = 1: điểm 998904999, thời gian 0.00s
k = 37: điểm 35173134821, thời gian 0.03s


k = 1000: điểm 203974744595, thời gian 0.51s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua tính vòng (chỉ giải bản tuyến tính)

In [14]:
def wrong_linear_only(nums, k):
    return pairs_linear(nums, k)

print("sai:", wrong_linear_only([1, 2, 3, 3], 2), "| đúng:", brute([1, 2, 3, 3], 2))

sai: 2 | đúng: 3


### Lỗi 2 — Cho một phần tử vừa đóng cặp vừa mở cặp mới (cập nhật tại chỗ)

Nếu cập nhật `free` rồi dùng ngay `free` mới để mở cặp tại cùng `x`, một phần tử thuộc hai đoạn.

In [15]:
def wrong_inplace(a, k):
    free = [0] + [NEG] * k; lng = [NEG] * (k + 1); sht = [NEG] * (k + 1)
    for x in a:
        for j in range(k, 0, -1):
            free[j] = max(free[j], lng[j - 1] + x, sht[j - 1] - x)
        for j in range(k + 1):
            lng[j] = max(lng[j], free[j] - x)                     # SAI: dùng free đã cập nhật bởi chính x
            sht[j] = max(sht[j], free[j] + x)
    return max(free)

print("sai:", wrong_inplace([1, 5, 1], 2), "| đúng:", pairs_linear([1, 5, 1], 2), "=", brute([1, 5, 1], 2))

sai: 8 | đúng: 4 = 4


### Lỗi 3 — Chỉ cho phép "mua trước bán sau" (quên bán khống)

Một đoạn có max đứng **trước** min vẫn có biên độ dương.

In [16]:
def wrong_long_only(a, k):
    free = [0] + [NEG] * k; lng = [NEG] * (k + 1)
    for x in a:
        nf = [free[0]] + [max(free[j], lng[j - 1] + x) for j in range(1, k + 1)]
        lng = [max(lng[j], free[j] - x) for j in range(k + 1)]; free = nf
    return max(free)

print("sai:", wrong_long_only([9, 1], 1), "| đúng:", brute([9, 1], 1))

sai: 0 | đúng: 8


### Lỗi 4 — Chỉ cắt vòng **trước** phần tử lớn nhất

Cần thử cả cắt **sau** max (khi min của đoạn chứa max nằm bên trái max). Tìm phản ví dụ bằng tìm kiếm ngẫu nhiên:

In [17]:
def wrong_one_rotation(nums, k):
    m = nums.index(max(nums))
    return pairs_linear(nums[m:] + nums[:m], k)

rng = random.Random(13)
found = None
while found is None:
    n = rng.randint(2, 7); k = rng.randint(1, n); a = [rng.randint(1, 6) for _ in range(n)]
    if wrong_one_rotation(a, k) != brute(a, k):
        found = (a, k)
a, k = found
print("phản ví dụ:", a, "k =", k, "| sai:", wrong_one_rotation(a, k), "| đúng:", brute(a, k))

phản ví dụ: [6, 5, 1, 3, 2] k = 2 | sai: 6 | đúng: 8


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng (max − min) qua phân hoạch, **tối đa**" ⇒ nới lỏng thành chọn cặp `(+x, −y)` ⇒ DP máy trạng thái kiểu cổ phiếu.
- Mảng vòng ⇒ tìm một phần tử **chắc chắn** có thể làm điểm cắt (ở đây là cạnh trước/sau max).
- Khi cập nhật DP theo phần tử, luôn tính trạng thái mới từ trạng thái **cũ** nếu mỗi phần tử chỉ được dùng một lần.
- Tối đa `k` đoạn mà thêm đoạn không bao giờ làm giảm (phân hoạch mịn hơn có thể tăng hoặc giảm) ⇒ lấy `max` trên mọi `j ≤ k`.

## 15. Biến thể của bài

1. **Bản tuyến tính** (không vòng): chỉ chạy `pairs_linear` một lần.
2. **Tối thiểu** tổng biên độ với đúng `k` đoạn: không nới lỏng được (min của max) — cần DP phân hoạch trực tiếp hoặc tham lam cắt tại `k − 1` khoảng chênh lớn nhất của mảng **đã sắp**? (chỉ đúng khi được sắp xếp lại).
3. **Best Time to Buy and Sell Stock IV** (LC 188): chỉ có `long`.
4. **Điểm = tổng (max − min)²**: không nới lỏng được, cần DP phân hoạch.

In [18]:
# Biến thể 3: LC 188 bằng cùng khung máy trạng thái (chỉ mua trước bán sau)
def stock_iv(k, prices):
    free = [0] + [NEG] * k; hold = [NEG] * (k + 1)
    for x in prices:
        nf = [free[0]] + [max(free[j], hold[j - 1] + x) for j in range(1, k + 1)]
        hold = [max(hold[j], free[j] - x) for j in range(k + 1)]; free = nf
    return max(free)

def stock_brute(k, p):
    from functools import lru_cache
    @lru_cache(None)
    def go(i, left, holding):
        if i == len(p): return 0 if holding is None else NEG
        best = go(i + 1, left, holding)
        if holding is None and left > 0: best = max(best, go(i + 1, left, p[i]))
        if holding is not None: best = max(best, p[i] - holding + go(i + 1, left - 1, None))
        return best
    return go(0, k, None)

rng = random.Random(15)
for _ in range(300):
    p = [rng.randint(1, 9) for _ in range(rng.randint(1, 8))]; k = rng.randint(1, 4)
    assert stock_iv(k, p) == stock_brute(k, p)
print("Biến thể 3 OK; LC188 ví dụ:", stock_iv(2, [3, 2, 6, 5, 0, 3]))

Biến thể 3 OK; LC188 ví dụ: 7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi tập cắt | `O(2^n n)` | Kiểm chứng |
| 2. DP phân hoạch × n | Trực tiếp | `O(n³ k)` | Kiểm chứng cỡ vừa |
| 3. Cặp × n | Nới lỏng | `O(n² k)` | Khi chưa thấy mẹo cắt |
| 4. Cặp × 2 | Nới lỏng + cắt tại max | `O(n k)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maximumScore([3,1,4,1,5], 1)` = ?

**Bài 2 (tính tay).** `maximumScore([1,5,1,5], 2)`.

**Bài 3 (điền TODO).** Viết `linear_partition_score(nums, k)` cho bản **tuyến tính**.

**Bài 4 (tự viết lại).** Thay vì cắt tại max, hãy cắt tại phần tử **nhỏ nhất** và chứng minh/kiểm chứng vẫn đúng.

**Bài 5 (mini challenge).** Trả về luôn một **phân hoạch tối ưu** (danh sách đoạn theo chỉ số vòng) cho bản tuyến tính.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def linear_partition_score(nums, k):
    # TODO: pairs_linear
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def linear_partition_with_cuts(nums, k):
    # TODO: DP phân hoạch trực tiếp + truy vết
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `k = 1` ⇒ một đoạn là cả vòng. (2) `max − min`. (3) `4`.

**Bài 2.** (1) Hai cặp `(5,1)`. (2) Mỗi cặp góp 4. (3) `8`.

**Bài 3.** (1) Không cần quay. (2) Gọi `pairs_linear(nums, k)`. (3) So với DP phân hoạch trực tiếp.

**Bài 4.** (1) Đối xứng: đổi dấu mọi phần tử, max thành min. (2) Biên độ không đổi khi đổi dấu. (3) Kiểm chứng ngẫu nhiên.

**Bài 5.** (1) `dp[t][i]` và `arg[t][i]`. (2) Truy vết từ `(t*, n)`. (3) Kiểm tra tổng biên độ các đoạn bằng điểm.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().maximumScore([3, 1, 4, 1, 5], 1) == 4
assert Solution().maximumScore([1, 5, 1, 5], 2) == 8 == brute([1, 5, 1, 5], 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def linear_partition_score_sol(nums, k):
    return pairs_linear(nums, k)

def cut_at_min(nums, k):
    m = nums.index(min(nums))
    return max(pairs_linear(nums[m:] + nums[:m], k), pairs_linear(nums[m + 1:] + nums[:m + 1], k))

rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 8); k = rng.randint(1, n); a = [rng.randint(1, 6) for _ in range(n)]
    assert linear_partition_score_sol(a, k) == partition_dp_linear(a, k)
    assert cut_at_min(a, k) == brute(a, k)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5: phân hoạch tối ưu (bản tuyến tính) bằng DP trực tiếp + truy vết
def linear_partition_with_cuts_sol(nums, k):
    n = len(nums)
    dp = [[NEG] * (n + 1) for _ in range(k + 1)]; arg = [[-1] * (n + 1) for _ in range(k + 1)]
    dp[0][0] = 0
    for t in range(1, k + 1):
        for i in range(1, n + 1):
            mx = mn = nums[i - 1]
            for j in range(i - 1, -1, -1):
                mx = max(mx, nums[j]); mn = min(mn, nums[j])
                if dp[t - 1][j] > NEG and dp[t - 1][j] + mx - mn > dp[t][i]:
                    dp[t][i] = dp[t - 1][j] + mx - mn; arg[t][i] = j
    t = max(range(1, k + 1), key=lambda t: dp[t][n]); best = dp[t][n]
    segs = []; i = n
    while t > 0:
        j = arg[t][i]; segs.append((j, i - 1)); i = j; t -= 1
    return best, segs[::-1]

rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 8); k = rng.randint(1, n); a = [rng.randint(1, 9) for _ in range(n)]
    best, segs = linear_partition_with_cuts_sol(a, k)
    assert len(segs) <= k and segs[0][0] == 0 and segs[-1][1] == n - 1
    assert sum(max(a[l:r + 1]) - min(a[l:r + 1]) for l, r in segs) == best == pairs_linear(a, k)
print("Bài 5 OK; ví dụ:", linear_partition_with_cuts_sol([2, 3, 3, 1], 2))

Bài 5 OK; ví dụ: (3, [(0, 1), (2, 3)])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def linear(a):                       # tối đa k cặp không đan xen, mỗi cặp |x - y|
    free = [0] + [-inf]*k; lng = [-inf]*(k+1); sht = [-inf]*(k+1)
    for x in a:
        nf  = [0] + [max(free[j], lng[j-1] + x, sht[j-1] - x) for j in 1..k]
        lng = [max(lng[j], free[j] - x)]; sht = [max(sht[j], free[j] + x)]; free = nf
    return max(free)
m = argmax(nums)
ans = max(linear(nums[m:] + nums[:m]), linear(nums[m+1:] + nums[:m+1]))
```

**Checklist:**

- [ ] Xử lý vòng bằng **hai** phép quay (trước và sau max)?
- [ ] Có cả trạng thái bán khống (`short`)?
- [ ] Trạng thái mới tính từ trạng thái cũ (một phần tử một hành động)?
- [ ] Lấy `max` trên mọi `j ≤ k`?
- [ ] Đã so với vét cạn mọi tập cắt?